# MATNETS MatCNN + CapsNet on MNIST — Kaggle 2×GPU
Runs two models back-to-back on MNIST using JAX `pmap` across 2 GPUs, then plots their accuracy curves together.

In [ ]:
# !git clone https://github.com/dsainvg/MATNETS.git
# !pip install -e ./MATNETS
import subprocess
import sys
import importlib

# 1. Install the package
subprocess.check_call([sys.executable, "-m", "pip", "install", "git+https://github.com/dsainvg/MATNETS.git"])

# 2. Force the site-packages to be re-evaluated
import site
importlib.reload(site)

# 3. Now try to import
import matnets

In [ ]:
import functools
import time
import jax
import jax.numpy as jnp
import numpy as np
import optax
import flax.linen as nn
from flax.training.train_state import TrainState
from flax import jax_utils
import matplotlib.pyplot as plt
import matnets as mtn

NUM_DEVICES = jax.local_device_count()
print("Devices:", jax.devices())

## Shared Config & Data

In [ ]:
EPOCHS        = 25
LEARNING_RATE = 0.001
SUBSET_SIZE   = 65536
NUM_CLASSES   = 10

# Scaled-down hyper-parameters to ~2M
CONV_FEATURES    = 96
PRIMARY_CHANNELS = 96
PRIMARY_DIM      = 16
DIGIT_DIM        = 16
DECODER_HIDDEN1  = 512
DECODER_HIDDEN2  = 512

# Batch size: 256/device → 512 total on 2 GPUs
PER_DEVICE_BATCH = 256
TOTAL_BATCH      = PER_DEVICE_BATCH * NUM_DEVICES
print(f"{NUM_DEVICES} device(s), {PER_DEVICE_BATCH}/device = {TOTAL_BATCH} total batch")

In [ ]:
from tensorflow.keras.datasets import mnist as keras_mnist

(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = keras_mnist.load_data()

# Standard float images (N, 28, 28, 1) for CapsNet
x_train_std = (x_train_raw[:SUBSET_SIZE, ..., None].astype(np.float32) / 255.0)
x_test_std  = (x_test_raw[..., None].astype(np.float32) / 255.0)
y_train_raw = y_train_raw[:SUBSET_SIZE]
y_train_oh  = np.eye(NUM_CLASSES, dtype=np.float32)[y_train_raw]
y_test_oh   = np.eye(NUM_CLASSES, dtype=np.float32)[y_test_raw]

# Matrix-embedded images (N, 28, 28, 1, 4, 4) for MatCNN
# Pixel value is broadcast to every entry of the n×n matrix.
MAT_N = 4
from numpy.lib.stride_tricks import sliding_window_view

def embed_pixels(imgs, n=MAT_N):
    """(N, H, W, C) → (N, H, W, C, n, n)  — extract n x n local neighborhood padded with zeros."""
    pad_top = n // 2
    pad_bot = (n - 1) // 2
    pad_l = n // 2
    pad_r = (n - 1) // 2
    padded = np.pad(imgs, ((0,0), (pad_top, pad_bot), (pad_l, pad_r), (0,0)), mode='constant')
    return sliding_window_view(padded, (n, n), axis=(1, 2)).copy()

def embed_pixels0(imgs, n=MAT_N):
    def _interleaved_axis_order(size, n):
        block_end = (size // n) * n
        order = np.arange(block_end).reshape(-1, n).T.reshape(-1)
        if block_end < size:
            order = np.concatenate((order, np.arange(block_end, size)))
        return order

    """(N, H, W, C) → (N, H, W, C, n, n)  — extract n x n local neighborhood padded with zeros."""
    pad_top = n // 2
    pad_bot = (n - 1) // 2
    pad_l = n // 2
    pad_r = (n - 1) // 2
    padded = np.pad(imgs, ((0,0), (pad_top, pad_bot), (pad_l, pad_r), (0,0)), mode='constant')
    windows = sliding_window_view(padded, (n, n), axis=(1, 2))
    row_order = _interleaved_axis_order(windows.shape[1], n)
    col_order = _interleaved_axis_order(windows.shape[2], n)
    return windows[:, row_order][:, :, col_order].copy()


x_train_mat = embed_pixels(x_train_std)   # (N, 28, 28, 1, 4, 4)
x_test_mat  = embed_pixels(x_test_std)

x_train_mat0 = embed_pixels0(x_train_std)   # (N, 28, 28, 1, 4, 4)
x_test_mat0  = embed_pixels0(x_test_std)

print("std shape:", x_train_std.shape, "  mat shape:", x_train_mat.shape)
print("mat0 shape:", x_train_mat0.shape)

def make_batches(x, y):
    """Yield (num_devices, per_device_batch, ...) shards, dropping remainder."""
    n = (len(x) // TOTAL_BATCH) * TOTAL_BATCH
    for i in range(0, n, TOTAL_BATCH):
        bx = x[i:i+TOTAL_BATCH].reshape(NUM_DEVICES, PER_DEVICE_BATCH, *x.shape[1:])
        by = y[i:i+TOTAL_BATCH].reshape(NUM_DEVICES, PER_DEVICE_BATCH, *y.shape[1:])
        yield jnp.array(bx), jnp.array(by)

---
## Part 1 — MatCNN (MATNETS matrix-neuron CNN)

In [ ]:
# 5 conv layers + 3 dense hidden layers. ~2M params, spread across layers.
# Conv: 16→32→64→64→64 (strides at c2,c3). Dense: 128→128→128.
C1, C2, C3, C4, C5 = 64//MAT_N, 128//MAT_N, 256//MAT_N, 256//MAT_N, 256//MAT_N
H1, H2, H3 = 128, 64, 32
print(f"MatCNN: C={C1},{C2},{C3},{C4},{C5}  H={H1},{H2},{H3}")

class MatCNN(nn.Module):
    c1:int=C1; c2:int=C2; c3:int=C3; c4:int=C4; c5:int=C5
    h1:int=H1; h2:int=H2; h3:int=H3
    n:int=MAT_N; num_classes:int=NUM_CLASSES

    @nn.compact
    def __call__(self, img):
        n = self.n
        def mp(name, q, p): return mtn.MatrixParams(
            W=self.param(f"{name}_W", nn.initializers.lecun_normal(), (q, p, 3, 3, n, n)),
            B=self.param(f"{name}_B", nn.initializers.zeros, (q, n, n)),
        )
        def dp(name, q, p): return mtn.MatrixParams(
            W=self.param(f"{name}_W", nn.initializers.lecun_normal(), (q, p, n, n)),
            B=self.param(f"{name}_B", nn.initializers.zeros, (q, n, n)),
        )
        h = jax.nn.relu(mtn.lax.matrix_conv2d(mp("c1", self.c1, 1),   img, padding="SAME"))
        h = jax.nn.relu(mtn.lax.matrix_conv2d(mp("c2", self.c2, self.c1), h, stride=2, padding="SAME"))
        h = jax.nn.relu(mtn.lax.matrix_conv2d(mp("c3", self.c3, self.c2), h, stride=2, padding="SAME"))
        h = jax.nn.relu(mtn.lax.matrix_conv2d(mp("c4", self.c4, self.c3), h, padding="SAME"))
        h = jax.nn.relu(mtn.lax.matrix_conv2d(mp("c5", self.c5, self.c4), h, padding="SAME"))
        h = h.mean(axis=(0, 1))
        h = mtn.dense(dp("d1", self.h1, self.c5), h, activation=jax.nn.relu)
        h = mtn.dense(dp("d2", self.h2, self.h1), h, activation=jax.nn.relu)
        h = mtn.dense(dp("d3", self.h3, self.h2), h, activation=jax.nn.relu)
        return mtn.dense(dp("out", self.num_classes, self.h3), h).mean(axis=(-2, -1))

mat_model = MatCNN()
mat_params = mat_model.init(jax.random.key(0), jnp.zeros((28, 28, 1, MAT_N, MAT_N)))
print(f"MatCNN params: {sum(x.size for x in jax.tree_util.tree_leaves(mat_params)):,}")


In [ ]:
mat_state = jax.device_put_replicated(
    TrainState.create(apply_fn=mat_model.apply, params=mat_params, tx=optax.adam(LEARNING_RATE)),
    jax.devices(),
)

def mat_loss_fn(params, bx, by):
    logits = jax.vmap(lambda img: mat_model.apply(params, img))(bx)
    return jnp.mean(optax.softmax_cross_entropy(logits=logits, labels=by))

@functools.partial(jax.pmap, axis_name="d")
def mat_train_step(state, bx, by):
    loss, grads = jax.value_and_grad(mat_loss_fn)(state.params, bx, by)
    grads = jax.lax.pmean(grads, "d")
    loss  = jax.lax.pmean(loss,  "d")
    return state.apply_gradients(grads=grads), loss

@functools.partial(jax.pmap, axis_name="d")
def mat_eval_step(state, bx, by):
    logits = jax.vmap(lambda img: mat_model.apply(state.params, img))(bx)
    acc = jnp.mean(jnp.argmax(logits, -1) == jnp.argmax(by, -1))
    return jax.lax.pmean(acc, "d")

mat_history = {"loss": [], "acc": []}

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    perm = np.random.permutation(len(x_train_mat))
    losses = []
    for bx, by in make_batches(x_train_mat[perm], y_train_oh[perm]):
        mat_state, loss = mat_train_step(mat_state, bx, by)
        losses.append(float(loss[0]))
    accs = [float(mat_eval_step(mat_state, bx, by)[0])
            for bx, by in make_batches(x_test_mat, y_test_oh)]
    mat_history["loss"].append(np.mean(losses))
    mat_history["acc"].append(np.mean(accs))
    print(f"[MatCNN] Epoch {epoch:2d}/{EPOCHS}  loss={mat_history['loss'][-1]:.4f}  "
          f"test_acc={mat_history['acc'][-1]*100:.2f}%  ({time.time()-t0:.1f}s)")

---
## Part 1.5 — MatCNN0 (MATNETS matrix-neuron CNN)

In [ ]:
# 5 conv layers + 3 dense hidden layers. ~2M params, spread across layers.
# Conv: 16→32→64→64→64 (strides at c2,c3). Dense: 128→128→128.
C1, C2, C3, C4, C5 = 64//MAT_N, 128//MAT_N, 256//MAT_N, 256//MAT_N, 256//MAT_N
H1, H2, H3 = 128, 64, 32
print(f"MatCNN0: C={C1},{C2},{C3},{C4},{C5}  H={H1},{H2},{H3}")

class MatCNN0(nn.Module):
    c1:int=C1; c2:int=C2; c3:int=C3; c4:int=C4; c5:int=C5
    h1:int=H1; h2:int=H2; h3:int=H3
    n:int=MAT_N; num_classes:int=NUM_CLASSES

    @nn.compact
    def __call__(self, img):
        n = self.n
        def mp(name, q, p): return mtn.MatrixParams(
            W=self.param(f"{name}_W", nn.initializers.lecun_normal(), (q, p, 3, 3, n, n)),
            B=self.param(f"{name}_B", nn.initializers.zeros, (q, n, n)),
        )
        def dp(name, q, p): return mtn.MatrixParams(
            W=self.param(f"{name}_W", nn.initializers.lecun_normal(), (q, p, n, n)),
            B=self.param(f"{name}_B", nn.initializers.zeros, (q, n, n)),
        )
        h = jax.nn.relu(mtn.lax.matrix_conv2d(mp("c1", self.c1, 1),   img, padding="SAME"))
        h = jax.nn.relu(mtn.lax.matrix_conv2d(mp("c2", self.c2, self.c1), h, stride=2, padding="SAME"))
        h = jax.nn.relu(mtn.lax.matrix_conv2d(mp("c3", self.c3, self.c2), h, stride=2, padding="SAME"))
        h = jax.nn.relu(mtn.lax.matrix_conv2d(mp("c4", self.c4, self.c3), h, padding="SAME"))
        h = jax.nn.relu(mtn.lax.matrix_conv2d(mp("c5", self.c5, self.c4), h, padding="SAME"))
        h = h.mean(axis=(0, 1))
        h = mtn.dense(dp("d1", self.h1, self.c5), h, activation=jax.nn.relu)
        h = mtn.dense(dp("d2", self.h2, self.h1), h, activation=jax.nn.relu)
        h = mtn.dense(dp("d3", self.h3, self.h2), h, activation=jax.nn.relu)
        return mtn.dense(dp("out", self.num_classes, self.h3), h).mean(axis=(-2, -1))

mat0_model = MatCNN0()
mat0_params = mat0_model.init(jax.random.key(0), jnp.zeros((28, 28, 1, MAT_N, MAT_N)))
print(f"MatCNN0 params: {sum(x.size for x in jax.tree_util.tree_leaves(mat0_params)):,}")


In [ ]:
mat0_state = jax.device_put_replicated(
    TrainState.create(apply_fn=mat0_model.apply, params=mat0_params, tx=optax.adam(LEARNING_RATE)),
    jax.devices(),
)

def mat0_loss_fn(params, bx, by):
    logits = jax.vmap(lambda img: mat0_model.apply(params, img))(bx)
    return jnp.mean(optax.softmax_cross_entropy(logits=logits, labels=by))

@functools.partial(jax.pmap, axis_name="d")
def mat0_train_step(state, bx, by):
    loss, grads = jax.value_and_grad(mat0_loss_fn)(state.params, bx, by)
    grads = jax.lax.pmean(grads, "d")
    loss  = jax.lax.pmean(loss,  "d")
    return state.apply_gradients(grads=grads), loss

@functools.partial(jax.pmap, axis_name="d")
def mat0_eval_step(state, bx, by):
    logits = jax.vmap(lambda img: mat0_model.apply(state.params, img))(bx)
    acc = jnp.mean(jnp.argmax(logits, -1) == jnp.argmax(by, -1))
    return jax.lax.pmean(acc, "d")

mat0_history = {"loss": [], "acc": []}

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    perm = np.random.permutation(len(x_train_mat0))
    losses = []
    for bx, by in make_batches(x_train_mat0[perm], y_train_oh[perm]):
        mat0_state, loss = mat0_train_step(mat0_state, bx, by)
        losses.append(float(loss[0]))
    accs = [float(mat0_eval_step(mat0_state, bx, by)[0])
            for bx, by in make_batches(x_test_mat0, y_test_oh)]
    mat0_history["loss"].append(np.mean(losses))
    mat0_history["acc"].append(np.mean(accs))
    print(f"[MatCNN0] Epoch {epoch:2d}/{EPOCHS}  loss={mat0_history['loss'][-1]:.4f}  "
          f"test_acc={mat0_history['acc'][-1]*100:.2f}%  ({time.time()-t0:.1f}s)")

---
## Part 2 — CapsNet

In [ ]:
def squash(x, axis=-1, eps=1e-7):
    sq = jnp.sum(jnp.square(x), axis=axis, keepdims=True)
    return (sq / (1.0 + sq)) * (x / jnp.sqrt(sq + eps))

def margin_loss(labels, lengths, m_plus=0.9, m_minus=0.1, lam=0.5):
    loss = labels * jnp.square(jnp.maximum(0., m_plus - lengths)) \
         + lam * (1 - labels) * jnp.square(jnp.maximum(0., lengths - m_minus))
    return jnp.mean(jnp.sum(loss, axis=-1))

def recon_loss(images, recons):
    return jnp.mean(jnp.sum(jnp.square(images.reshape(images.shape[0], -1)
                                       - recons.reshape(recons.shape[0], -1)), axis=-1))

class PrimaryCaps(nn.Module):
    channels: int; capsule_dim: int

    @nn.compact
    def __call__(self, x):
        x = nn.Conv(self.channels, (9, 9), strides=(2, 2), padding="VALID")(x)
        B = x.shape[0]
        return squash(x.reshape(B, -1, self.capsule_dim))

class DigitCaps(nn.Module):
    num_caps: int = 10; cap_dim: int = 32; routings: int = 3

    @nn.compact
    def __call__(self, x):
        B, num_primary, pdim = x.shape
        W = self.param("W", nn.initializers.glorot_uniform(),
                       (num_primary, self.num_caps, pdim, self.cap_dim))
        u_hat = jnp.einsum("bie,ijed->bijd", x, W)
        b = jnp.zeros((B, num_primary, self.num_caps))
        for i in range(self.routings):
            c = jax.nn.softmax(b, axis=-1)
            v = squash(jnp.sum(jnp.expand_dims(c, -1) * u_hat, axis=1))
            if i < self.routings - 1:
                b = b + jnp.einsum("bijd,bjd->bij", u_hat, v)
        return v

class CapsNet(nn.Module):
    conv_features: int = CONV_FEATURES
    primary_channels: int = PRIMARY_CHANNELS
    primary_dim: int = PRIMARY_DIM
    digit_dim: int = DIGIT_DIM
    dec_h1: int = DECODER_HIDDEN1
    dec_h2: int = DECODER_HIDDEN2
    num_classes: int = NUM_CLASSES

    @nn.compact
    def __call__(self, x, labels=None):
        B = x.shape[0]
        orig_shape = x.shape[1:]
        h = nn.relu(nn.Conv(self.conv_features, (9, 9), padding="VALID")(x))
        h = PrimaryCaps(self.primary_channels, self.primary_dim)(h)
        caps = DigitCaps(self.num_classes, self.digit_dim)(h)
        lengths = jnp.sqrt(jnp.sum(jnp.square(caps), axis=-1) + 1e-7)
        mask = labels if labels is not None else jax.nn.one_hot(jnp.argmax(lengths, -1), self.num_classes)
        dec_in = (caps * jnp.expand_dims(mask, -1)).reshape(B, -1)
        dec_in = nn.relu(nn.Dense(self.dec_h1)(dec_in))
        dec_in = nn.relu(nn.Dense(self.dec_h2)(dec_in))
        recon  = nn.sigmoid(nn.Dense(int(np.prod(orig_shape)))(dec_in)).reshape(B, *orig_shape)
        return lengths, recon

caps_model = CapsNet()
dummy_b1 = jnp.zeros((1, 28, 28, 1))
caps_vars = caps_model.init(jax.random.key(1), dummy_b1, jnp.zeros((1, NUM_CLASSES)))
print(f"CapsNet params: {sum(x.size for x in jax.tree_util.tree_leaves(caps_vars)):,}")

In [ ]:
caps_state = jax_utils.replicate(
    TrainState.create(apply_fn=caps_model.apply,
                      params=caps_vars["params"],
                      tx=optax.adam(LEARNING_RATE))
)

ALPHA = 0.0005

def caps_loss_fn(params, bx, by):
    lengths, recon = caps_model.apply({"params": params}, bx, labels=by)
    return margin_loss(by, lengths) + ALPHA * recon_loss(bx, recon)

@functools.partial(jax.pmap, axis_name="d")
def caps_train_step(state, bx, by):
    loss, grads = jax.value_and_grad(caps_loss_fn)(state.params, bx, by)
    grads = jax.lax.pmean(grads, "d")
    loss  = jax.lax.pmean(loss,  "d")
    return state.apply_gradients(grads=grads), loss

@functools.partial(jax.pmap, axis_name="d")
def caps_eval_step(state, bx, by):
    lengths, _ = caps_model.apply({"params": state.params}, bx)
    acc = jnp.mean(jnp.argmax(lengths, -1) == jnp.argmax(by, -1))
    return jax.lax.pmean(acc, "d")

caps_history = {"loss": [], "acc": []}

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    perm = np.random.permutation(len(x_train_std))
    losses = []
    for bx, by in make_batches(x_train_std[perm], y_train_oh[perm]):
        caps_state, loss = caps_train_step(caps_state, bx, by)
        losses.append(float(loss[0]))
    accs = [float(caps_eval_step(caps_state, bx, by)[0])
            for bx, by in make_batches(x_test_std, y_test_oh)]
    caps_history["loss"].append(np.mean(losses))
    caps_history["acc"].append(np.mean(accs))
    print(f"[CapsNet] Epoch {epoch:2d}/{EPOCHS}  loss={caps_history['loss'][-1]:.4f}  "
          f"test_acc={caps_history['acc'][-1]*100:.2f}%  ({time.time()-t0:.1f}s)")

---
## Part 3 — Standard CNN (Flax)

In [ ]:
# 5 conv layers + 3 dense hidden layers. ~2M params, spread across layers.
# Conv: 32→64→128→256→256 (strides at c2,c3). Dense: 512→1024→512.
C1_std, C2_std, C3_std, C4_std, C5_std = 32, 64, 128, 256, 256
H1_std, H2_std, H3_std = 512, 1024, 512

class StandardCNN(nn.Module):
    @nn.compact
    def __call__(self, x):   # x: (H, W, 1)
        x = nn.relu(nn.Conv(C1_std, (3,3), padding="SAME")(x))
        x = nn.relu(nn.Conv(C2_std, (3,3), strides=(2,2), padding="SAME")(x))
        x = nn.relu(nn.Conv(C3_std, (3,3), strides=(2,2), padding="SAME")(x))
        x = nn.relu(nn.Conv(C4_std, (3,3), padding="SAME")(x))
        x = nn.relu(nn.Conv(C5_std, (3,3), padding="SAME")(x))
        x = x.mean(axis=(0, 1))
        x = nn.relu(nn.Dense(H1_std)(x))
        x = nn.relu(nn.Dense(H2_std)(x))
        x = nn.relu(nn.Dense(H3_std)(x))
        return nn.Dense(NUM_CLASSES)(x)

cnn_model = StandardCNN()
cnn_params = cnn_model.init(jax.random.key(2), jnp.zeros((28, 28, 1)))
print(f"StandardCNN params: {sum(v.size for v in jax.tree_util.tree_leaves(cnn_params)):,}")

In [ ]:
cnn_state = jax.device_put_replicated(
    TrainState.create(apply_fn=cnn_model.apply, params=cnn_params, tx=optax.adam(LEARNING_RATE)),
    jax.devices(),
)

def cnn_loss_fn(params, bx, by):
    logits = jax.vmap(lambda img: cnn_model.apply(params, img))(bx)
    return jnp.mean(optax.softmax_cross_entropy(logits=logits, labels=by))

@functools.partial(jax.pmap, axis_name="d")
def cnn_train_step(state, bx, by):
    loss, grads = jax.value_and_grad(cnn_loss_fn)(state.params, bx, by)
    grads = jax.lax.pmean(grads, "d")
    loss  = jax.lax.pmean(loss,  "d")
    return state.apply_gradients(grads=grads), loss

@functools.partial(jax.pmap, axis_name="d")
def cnn_eval_step(state, bx, by):
    logits = jax.vmap(lambda img: cnn_model.apply(state.params, img))(bx)
    acc = jnp.mean(jnp.argmax(logits, -1) == jnp.argmax(by, -1))
    return jax.lax.pmean(acc, "d")

cnn_history = {"loss": [], "acc": []}

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    perm = np.random.permutation(len(x_train_std))
    losses = []
    for bx, by in make_batches(x_train_std[perm], y_train_oh[perm]):
        cnn_state, loss = cnn_train_step(cnn_state, bx, by)
        losses.append(float(loss[0]))
    accs = [float(cnn_eval_step(cnn_state, bx, by)[0])
            for bx, by in make_batches(x_test_std, y_test_oh)]
    cnn_history["loss"].append(np.mean(losses))
    cnn_history["acc"].append(np.mean(accs))
    print(f"[StdCNN]  Epoch {epoch:2d}/{EPOCHS}  loss={cnn_history['loss'][-1]:.4f}  "
          f"test_acc={cnn_history['acc'][-1]*100:.2f}%  ({time.time()-t0:.1f}s)")

---
## Accuracy & Loss Curves — All Three Models

In [ ]:
epochs = range(1, EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(epochs, [a*100 for a in mat_history["acc"]],  marker="o", label="MatCNN")
ax1.plot(epochs, [a*100 for a in mat0_history["acc"]],  marker="d", label="MatCNN0")
ax1.plot(epochs, [a*100 for a in caps_history["acc"]], marker="s", label="CapsNet")
ax1.plot(epochs, [a*100 for a in cnn_history["acc"]],  marker="^", label="StdCNN")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Test Accuracy (%)")
ax1.set_title("Test Accuracy"); ax1.legend(); ax1.grid(True)

ax2.plot(epochs, mat_history["loss"],  marker="o", label="MatCNN")
ax2.plot(epochs, mat0_history["loss"],  marker="d", label="MatCNN0")
ax2.plot(epochs, caps_history["loss"], marker="s", label="CapsNet")
ax2.plot(epochs, cnn_history["loss"],  marker="^", label="StdCNN")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Train Loss")
ax2.set_title("Training Loss"); ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.savefig("accuracy_loss_curves.png", dpi=150)
plt.show()

print(f"\nFinal MatCNN  test acc: {mat_history['acc'][-1]*100:.2f}%")
print(f"\nFinal MatCNN0 test acc: {mat0_history['acc'][-1]*100:.2f}%")
print(f"Final CapsNet test acc: {caps_history['acc'][-1]*100:.2f}%")
print(f"Final StdCNN  test acc: {cnn_history['acc'][-1]*100:.2f}%")

In [ ]:
import pickle as pkl
with open("mnist_results.pkl", "wb") as f:
    pkl.dump({
        "mat_history": mat_history,
        "mat0_history": mat0_history,
        "caps_history": caps_history,
        "cnn_history": cnn_history,
    }, f)
